# Week 2-2 · metadata filter로 검색 범위 통제하기

## 시나리오
같은 단어가 여러 playbook에 있어도 분류된 category와 일치하는 문서만 검색 후보로 사용합니다.

## 학습 목표
- metadata가 있는 `Document` corpus를 만든다.
- 유사도 계산 전에 category filter를 적용한다.
- filter 유무에 따른 오인용 가능성을 비교한다.

## 직접 조립
완성된 `weekX.app` 함수를 가져오지 않습니다. 아래 코드에서 작은 fixture와 핵심 객체·함수·연결을 직접 만듭니다.

### 1단계 · playbook corpus

In [ ]:
# 실행 순서: 1단계 · playbook corpus에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 1단계 · playbook corpus.
from langchain_core.documents import Document

practice_playbooks = [
    Document(page_content="중복 결제는 거래 ID를 확인하고 billing 담당자에게 전달합니다.", metadata={"category": "billing", "chunk_id": "bill-01"}),
    Document(page_content="로그인 실패는 계정 잠금 상태를 확인하고 access 담당자에게 전달합니다.", metadata={"category": "access", "chunk_id": "access-01"}),
    Document(page_content="서비스 오류는 상태 페이지와 최근 배포를 확인합니다.", metadata={"category": "technical", "chunk_id": "tech-01"}),
]
[(doc.metadata["category"], doc.metadata["chunk_id"]) for doc in practice_playbooks]

### 2단계 · filter 우선 검색

In [ ]:
# 실행 순서: 2단계 · filter 우선 검색에서 metadata_filter, practice_retrieve_playbook을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 2단계 · filter 우선 검색.
# 분류 결과와 같은 metadata를 가진 문서만 검색 후보로 남깁니다.
def metadata_filter(doc: Document, expected: dict[str, str]) -> bool:
    return all(doc.metadata.get(key) == value for key, value in expected.items())

# metadata filter를 relevance 정렬보다 먼저 적용해 category 혼입을 막습니다.
def practice_retrieve_playbook(query: str, category: str) -> list[Document]:
    candidates = [doc for doc in practice_playbooks if metadata_filter(doc, {"category": category})]
    query_tokens = set(query.lower().split())
    return sorted(candidates, key=lambda doc: len(query_tokens & set(doc.page_content.lower().split())), reverse=True)

billing_results = practice_retrieve_playbook("중복 결제 로그인", "billing")
access_results = practice_retrieve_playbook("중복 결제 로그인", "access")
[(doc.metadata["category"], doc.metadata["chunk_id"]) for doc in billing_results + access_results]

### 3단계 · category 격리 확인

In [ ]:
# 실행 순서: 3단계 · category 격리 확인에서 fixture와 assertion을(를) 먼저 구성합니다.
# 관찰 포인트: 이 셀의 출력이 다음 단계에서 사용할 입력 계약을 충족하는지 확인합니다 — 3단계 · category 격리 확인.
missing_results = practice_retrieve_playbook("가격 문의", "other")
assert all(doc.metadata["category"] == "billing" for doc in billing_results)
assert all(doc.metadata["category"] == "access" for doc in access_results)
assert missing_results == []
{"billing": len(billing_results), "access": len(access_results), "other": len(missing_results)}

## 중간 결과
각 코드 셀의 출력에서 입력이 어떤 상태로 변했는지 확인합니다. 마지막 `assert`는 눈으로 본 결과를 실행 가능한 계약으로 고정합니다.

## 실패 경계
해당 category 문서가 없으면 다른 category를 빌려 쓰지 않고 빈 결과를 반환합니다.

## 실제 app 연결
Week 2 app의 PGVector retrieval은 `filter={"category": ...}`를 전달합니다. 이 실습은 DB 없이 filter가 후보 집합을 어떻게 좁히는지만 보여주며, lexical/embedding relevance 판정까지 충분히 구현한 검색기는 아닙니다.

### 확장 과제
fixture의 문장이나 임계값을 하나 바꾸고, 어느 중간 결과와 assertion이 달라지는지 기록하세요.

## 다음 Notebook 연결
다음 `03_conditional_resolution_routing.ipynb`에서는 분류 결과에 따라 실제 graph edge가 달라지게 만듭니다.